# IndPenSim fault-inclusive soft sensor — Colab and Jupyter

This notebook runs the fault-aware concentration model from the previous analysis. It is **self-contained**: you do not need to upload a separate `.py` file or execute the old notebook first. It runs the new normal-only versus fault-aware HGB comparison; the original Random Forest baseline and five follow-up experiments remain in your separate `baseline.ipynb`.

## Start here

1. Put `100_Batches_IndPenSim_V3.csv` in Google Drive, for example in `MyDrive/IndPenSim_Data/`.
2. Use a fresh Python 3 Colab session. A standard CPU runtime is sufficient; GPU acceleration is not used by this implementation.
3. Run Step 1 for this notebook's environment. Use Step 2 only to connect Drive in Colab; edit `DATA_PATH` in Step 3.
4. Choose `smoke` for a setup check or `full` for the complete experiment. Then run the remaining cells in order.
5. Review the results and download the ZIP. Each run saves into a new timestamped Drive folder.

The large raw CSV is not included in this notebook. It is read from your Drive; only the non-Raman process columns are loaded. Alternatively, `DATA_PATH` can point to the directory containing the four saved split CSVs from your previous analysis.

## What is being tested?

| Evaluation | Normal-only model | Fault-aware model | Held-out batches |
|---|---|---|---|
| Five normal-batch folds | 72 normal batches | The same 72 normals plus all 10 development faults | 18 normal batches per fold: 6 per regime |
| Leave-one-fault-batch-out | All 90 normal batches | The same 90 normals plus the other 9 faults | One complete fault batch per fold |

Every tested batch is excluded from fitting its own prediction model. The final saved bundle is then fitted on all 100 labelled batches for future application; its in-sample predictions are not a test of accuracy.

The target is **current penicillin concentration (g/L)**, not a future forecast. This is a research prototype, not a plant-control or safety system. Improvements are not guaranteed for every fault batch.

Repository copy: saved outputs and personal paths have been removed. Recorded full-run results are in `../results/ml4/`. Locally, install `requirements/main.txt` from the repository root before running the notebook. See `../docs/REPRODUCIBILITY.md`.


## Step 1 — Check or install this notebook's environment

Colab can install the required versions. Local Jupyter checks the environment without installing automatically; install from `requirements/main.txt` first. Restart the session/kernel if requested, then run from the beginning.


In [ ]:
# Repository setup: use this notebook's own environment
EXPECTED_VERSIONS = {'numpy': '2.3.5', 'scipy': '1.17.0', 'pandas': '2.2.3', 'scikit-learn': '1.8.0', 'matplotlib': '3.10.8', 'seaborn': '0.13.2', 'joblib': '1.5.3'}
REQUIREMENTS_FILE = 'requirements/main.txt'
import importlib.metadata
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.12 for this repository's documented environment.")

try:
    import google.colab
except ImportError:
    IN_GOOGLE_COLAB = False
else:
    IN_GOOGLE_COLAB = True

# Colab may install packages. Locally, install through the requirements
# file first; change this flag only if you intentionally want installation.
INSTALL_PACKAGES = IN_GOOGLE_COLAB
module_names = {"scikit-learn": "sklearn"}
installed_versions = {}
for package in EXPECTED_VERSIONS:
    try:
        installed_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_versions[package] = None
mismatches = {
    package: (installed_versions[package], expected)
    for package, expected in EXPECTED_VERSIONS.items()
    if installed_versions[package] != expected
}
loaded_versions = {
    package: getattr(sys.modules.get(module_names.get(package, package)), "__version__", None)
    for package in EXPECTED_VERSIONS
}

if mismatches and INSTALL_PACKAGES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        *[f"{package}=={version}" for package, version in EXPECTED_VERSIONS.items()],
    ])
elif mismatches:
    raise RuntimeError(
        f"Install this notebook's environment first: python -m pip install -r {REQUIREMENTS_FILE}. "
        "Use a separate environment from the other notebook. Mismatches: " + str(mismatches)
    )

stale = [
    package for package, loaded in loaded_versions.items()
    if loaded is not None and loaded != EXPECTED_VERSIONS[package]
]
if stale:
    raise RuntimeError(
        "Restart the Colab session or Jupyter kernel, then run from the beginning. "
        "Older packages remain loaded: " + ", ".join(stale)
    )

for package, expected in EXPECTED_VERSIONS.items():
    actual = importlib.metadata.version(package)
    if actual != expected:
        raise RuntimeError(f"Version mismatch after setup: {package} {actual}, expected {expected}.")
    print(package, actual)
print("Environment ready. Colab:", IN_GOOGLE_COLAB)
print("Setup did not train a model. Run the following cells in order.")


## Step 2 — Optional Google Drive connection

In Colab, keep mounting enabled if your dataset is on Drive. Set `MOUNT_GOOGLE_DRIVE = False` for session-local files. Local Jupyter skips this step. Never put account credentials in code.


In [ ]:
# Optional Google Drive connection; local Jupyter skips this step.
MOUNT_GOOGLE_DRIVE = True
if IN_GOOGLE_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive mount skipped. Use a local/session data path in the next cell.")


## Step 3 — Set your paths and run mode

Change `DATA_PATH` to the real location of your CSV. In Colab's left-hand Files panel, find the file, right-click it and choose **Copy path**. A directory input must contain all four files below:

- `train_normal_60_batches.csv`
- `validation_normal_15_batches.csv`
- `test_normal_15_batches.csv`
- `test_fault_10_batches.csv`

`smoke` evaluates one normal fold and fault batch 91 with 50 boosting iterations. It is only a software check. `full` evaluates all five normal folds and all ten fault folds with 180 iterations. Only the full results should be considered for your research report; they still require scientific review.

Keep the fault-batch training multiplier at 3.0 to match the previous implementation. Changing it is a new modelling experiment, not a formatting change.


In [ ]:
# Edit DATA_PATH and OUTPUT_ROOT to use your own data and output locations.
import os
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_GOOGLE_COLAB and MOUNT_GOOGLE_DRIVE:
    DEFAULT_DATA_PATH = Path("/content/drive/MyDrive/IndPenSim_Data/100_Batches_IndPenSim_V3.csv")
    DEFAULT_OUTPUT_ROOT = Path("/content/drive/MyDrive/IndPenSim_Results") / 'main'
else:
    DEFAULT_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "100_Batches_IndPenSim_V3.csv"
    DEFAULT_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / 'main'

DATA_PATH = os.environ.get("INDPENSIM_DATA_PATH", str(DEFAULT_DATA_PATH))
OUTPUT_ROOT = os.environ.get("INDPENSIM_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT))
RUN_MODE = "full"  # Choose "smoke" only for a technical setup check.

data_source = Path(DATA_PATH).expanduser()
if not data_source.exists():
    raise FileNotFoundError(
        f"Cannot find the data at: {data_source}\n"
        "Set DATA_PATH to the real dataset location and rerun this cell."
    )
if data_source.is_file() and data_source.suffix.lower() != ".csv":
    raise ValueError("Select the data CSV, not a ZIP archive or notebook.")
if data_source.is_dir():
    required_split_files = [
        "train_normal_60_batches.csv", "validation_normal_15_batches.csv",
        "test_normal_15_batches.csv", "test_fault_10_batches.csv",
    ]
    missing_split_files = [name for name in required_split_files if not (data_source / name).is_file()]
    if missing_split_files:
        raise FileNotFoundError("Missing split CSVs: " + ", ".join(missing_split_files))
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("RUN_MODE must be 'full' or 'smoke'.")

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
OUTPUT_DIR = Path(OUTPUT_ROOT).expanduser() / f"{RUN_MODE}_{run_stamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("Input:", data_source)
print("Mode:", RUN_MODE)
print("New output directory:", OUTPUT_DIR)
print("Full evaluation: 5 normal folds + 10 leave-one-fault-batch-out folds.")
print("Earlier saved results are not overwritten by this new run directory.")


## Step 4 — Load the complete model implementation

Run every cell from **4A to 4G**. They define the functions; model fitting starts in Step 5. Expand **Show code** on a collapsed cell to read its implementation.

The model keeps the same 36 inputs as the previous fault-aware program: 20 current/base measurements, 15 within-batch history features, and cumulative feed volume. The history uses only the current and previous observations. The target concentration, fault reference and batch identifiers are not predictive inputs.

The fault-risk classifier learns a separate label marking the period from fault onset onward. Its class-weighted score is not independently probability-calibrated. OOD is a normal-reference similarity warning. The empirical range is a heuristic based on out-of-fold errors, not a guaranteed confidence or safety interval.


In [ ]:
# @title 4A — Imports and the 36 feature definitions
from __future__ import annotations

import argparse
import json
import math
import platform
import sys
import time
from pathlib import Path
from typing import Any, Iterable, Sequence

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    IsolationForest,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42

TIME_COL = "Time (h)"
TARGET_COL = "Penicillin concentration(P:g/L)"
BATCH_COL = "Batch_ID"
FAULT_REF_COL = "Fault reference(Fault_ref:Fault ref)"

NORMAL_BATCHES = list(range(1, 91))
FAULT_BATCHES = list(range(91, 101))

BASE_FEATURES = [
    TIME_COL,
    "Aeration rate(Fg:L/h)",
    "Sugar feed rate(Fs:L/h)",
    "Acid flow rate(Fa:L/h)",
    "Base flow rate(Fb:L/h)",
    "Heating/cooling water flow rate(Fc:L/h)",
    "Heating water flow rate(Fh:L/h)",
    "Water for injection/dilution(Fw:L/h)",
    "Air head pressure(pressure:bar)",
    "Dissolved oxygen concentration(DO2:mg/L)",
    "Vessel Volume(V:L)",
    "Vessel Weight(Wt:Kg)",
    "pH(pH:pH)",
    "Temperature(T:K)",
    "Generated heat(Q:kJ)",
    "carbon dioxide percent in off-gas(CO2outgas:%)",
    "PAA flow(Fpaa:PAA flow (L/h))",
    "Oil flow(Foil:L/hr)",
    "Oxygen Uptake Rate(OUR:(g min^{-1}))",
    "Oxygen in percent in off-gas(O2:O2  (%))",
]

HISTORY_FEATURES = [
    "Dissolved oxygen concentration(DO2:mg/L)",
    "Sugar feed rate(Fs:L/h)",
    "Temperature(T:K)",
    "pH(pH:pH)",
    "Aeration rate(Fg:L/h)",
]

CUMULATIVE_FEED_COL = "Cumulative_Sugar_Feed"

ENGINEERED_FEATURES: list[str] = []
for _feature in HISTORY_FEATURES:
    ENGINEERED_FEATURES.extend(
        [
            f"{_feature}_lag1",
            f"{_feature}_difference1",
            f"{_feature}_previous5_mean",
        ]
    )
ENGINEERED_FEATURES.append(CUMULATIVE_FEED_COL)

MODEL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

# Time and cumulative feed are strong batch-progress variables.  They remain
# useful to the concentration regressor, but are omitted from the reliability
# models to make their warnings less dependent on batch age alone.
RELIABILITY_FEATURES = [
    feature
    for feature in MODEL_FEATURES
    if feature not in {TIME_COL, CUMULATIVE_FEED_COL}
]

SPLIT_FILENAMES = [
    "train_normal_60_batches.csv",
    "validation_normal_15_batches.csv",
    "test_normal_15_batches.csv",
    "test_fault_10_batches.csv",
]


In [ ]:
# @title 4B — Load data and build causal history features
def _numeric_name(name: str) -> bool:
    """Return True for Raman columns whose headings are numeric shifts."""
    try:
        float(str(name).strip())
        return True
    except ValueError:
        return False


def _read_one_csv(path: Path) -> pd.DataFrame:
    """Read process/meta columns while skipping the 2,200 Raman columns."""
    header = pd.read_csv(path, nrows=0).columns.tolist()
    process_columns = [column for column in header if not _numeric_name(column)]
    return pd.read_csv(path, usecols=process_columns, low_memory=False)


def load_indpensim(path: str | Path, require_target: bool = True) -> pd.DataFrame:
    """Load a raw IndPenSim CSV or the directory of four saved split CSVs."""
    source = Path(path).expanduser().resolve()

    if source.is_dir():
        missing = [name for name in SPLIT_FILENAMES if not (source / name).exists()]
        if missing:
            raise FileNotFoundError(
                "The data directory must contain the four split files. Missing: "
                + ", ".join(missing)
            )
        frame = pd.concat(
            [_read_one_csv(source / name) for name in SPLIT_FILENAMES],
            ignore_index=True,
        )
    elif source.is_file():
        frame = _read_one_csv(source)
    else:
        raise FileNotFoundError(f"Data path does not exist: {source}")

    required = [TIME_COL] + BASE_FEATURES[1:]
    if require_target:
        required.extend([TARGET_COL, FAULT_REF_COL])
    missing_columns = [column for column in required if column not in frame.columns]
    if missing_columns:
        raise ValueError(
            "Required IndPenSim columns are missing:\n- "
            + "\n- ".join(missing_columns)
        )

    numeric_columns = list(dict.fromkeys(required + [TARGET_COL, FAULT_REF_COL]))
    for column in numeric_columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    if BATCH_COL in frame.columns:
        frame[BATCH_COL] = pd.to_numeric(frame[BATCH_COL], errors="coerce")
        if frame[BATCH_COL].isna().any():
            raise ValueError(f"{BATCH_COL} contains missing or non-numeric values.")
        frame[BATCH_COL] = frame[BATCH_COL].astype(int)
    else:
        # The raw combined file begins a new batch whenever Time resets.
        reset = frame[TIME_COL].diff().lt(0).fillna(False)
        frame[BATCH_COL] = reset.cumsum().astype(int) + 1

    frame = frame.sort_values([BATCH_COL, TIME_COL], kind="stable").reset_index(drop=True)
    return frame


def validate_training_data(frame: pd.DataFrame) -> None:
    """Fail early if the expected 100-batch experimental design is absent."""
    observed = sorted(frame[BATCH_COL].unique().astype(int).tolist())
    expected = list(range(1, 101))
    if observed != expected:
        raise ValueError(
            "Full training/evaluation requires reconstructed batches 1--100. "
            f"Observed: {observed}"
        )
    if frame[TARGET_COL].notna().sum() == 0:
        raise ValueError("The target column has no observed values.")
    for batch in FAULT_BATCHES:
        active = (
            frame.loc[frame[BATCH_COL].eq(batch), FAULT_REF_COL]
            .fillna(0)
            .ne(0)
        )
        if not active.any():
            raise ValueError(
                f"Fault batch {batch} has no non-zero fault reference; "
                "fault-onset labels cannot be created."
            )


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create only causal, within-batch history features."""
    data = frame.copy()
    group = data.groupby(BATCH_COL, sort=False)

    for feature in HISTORY_FEATURES:
        data[f"{feature}_lag1"] = group[feature].shift(1)
        data[f"{feature}_difference1"] = group[feature].diff(1)
        data[f"{feature}_previous5_mean"] = group[feature].transform(
            lambda values: values.shift(1).rolling(5, min_periods=1).mean()
        )

    time_step = group[TIME_COL].diff().clip(lower=0).fillna(0)
    feed_added = data["Sugar feed rate(Fs:L/h)"].mul(time_step).fillna(0)
    data[CUMULATIVE_FEED_COL] = feed_added.groupby(data[BATCH_COL]).cumsum()

    data[MODEL_FEATURES] = data[MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
    return data


def add_fault_phase_labels(frame: pd.DataFrame) -> pd.DataFrame:
    """Label normal/pre-onset rows 0 and rows from a fault onset onward 1."""
    data = frame.copy()
    data["Fault_Affected"] = 0
    data["Fault_Phase"] = "Normal"

    for batch in FAULT_BATCHES:
        batch_rows = data[BATCH_COL].eq(batch)
        active = batch_rows & data[FAULT_REF_COL].fillna(0).ne(0)
        onset = float(data.loc[active, TIME_COL].min())
        after_onset = batch_rows & data[TIME_COL].ge(onset)
        before_onset = batch_rows & data[TIME_COL].lt(onset)
        data.loc[before_onset, "Fault_Phase"] = "Fault batch: before onset"
        data.loc[after_onset, "Fault_Phase"] = "Fault batch: onset/after"
        data.loc[after_onset, "Fault_Affected"] = 1

    return data


In [ ]:
# @title 4C — Define and fit the regression and fault-risk models
def build_regressor(max_iter: int, random_state: int) -> Pipeline:
    """Create an untuned, reproducible nonlinear regressor."""
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                HistGradientBoostingRegressor(
                    learning_rate=0.05,
                    max_iter=max_iter,
                    max_leaf_nodes=31,
                    min_samples_leaf=30,
                    l2_regularization=1.0,
                    early_stopping=False,
                    random_state=random_state,
                ),
            ),
        ]
    )


def build_risk_classifier(max_iter: int, random_state: int) -> Pipeline:
    """Create a classifier that estimates whether a row is fault affected."""
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=max_iter,
                    max_leaf_nodes=15,
                    min_samples_leaf=30,
                    l2_regularization=2.0,
                    early_stopping=False,
                    random_state=random_state,
                ),
            ),
        ]
    )


def batch_equal_regression_weights(
    frame: pd.DataFrame, fault_multiplier: float
) -> np.ndarray:
    """Give every batch equal mass, then upweight fault-development batches."""
    counts = frame.groupby(BATCH_COL)[BATCH_COL].transform("size").to_numpy(float)
    weights = 1.0 / counts
    is_fault = frame[BATCH_COL].isin(FAULT_BATCHES).to_numpy()
    weights[is_fault] *= fault_multiplier
    return weights / weights.mean()


def batch_and_class_balanced_weights(frame: pd.DataFrame) -> np.ndarray:
    """Balance batches first and positive/negative fault-phase labels second."""
    counts = frame.groupby(BATCH_COL)[BATCH_COL].transform("size").to_numpy(float)
    base = 1.0 / counts
    labels = frame["Fault_Affected"].to_numpy(int)
    mass_zero = float(base[labels == 0].sum())
    mass_one = float(base[labels == 1].sum())
    if mass_zero == 0 or mass_one == 0:
        raise ValueError("The risk-classifier training set must contain both classes.")
    total = mass_zero + mass_one
    multipliers = np.where(
        labels == 1,
        total / (2.0 * mass_one),
        total / (2.0 * mass_zero),
    )
    weights = base * multipliers
    return weights / weights.mean()


def fit_regressor(
    train: pd.DataFrame,
    max_iter: int,
    random_state: int,
    fault_multiplier: float,
) -> Pipeline:
    model = build_regressor(max_iter=max_iter, random_state=random_state)
    weights = batch_equal_regression_weights(train, fault_multiplier=fault_multiplier)
    model.fit(
        train[MODEL_FEATURES],
        train[TARGET_COL],
        model__sample_weight=weights,
    )
    return model


def fit_risk_classifier(
    train: pd.DataFrame, max_iter: int, random_state: int
) -> Pipeline:
    model = build_risk_classifier(max_iter=max_iter, random_state=random_state)
    weights = batch_and_class_balanced_weights(train)
    model.fit(
        train[RELIABILITY_FEATURES],
        train["Fault_Affected"],
        model__sample_weight=weights,
    )
    return model


In [ ]:
# @title 4D — Create complete-batch folds and held-out predictions
def regime(batch: int) -> str:
    if 1 <= batch <= 30:
        return "Recipe"
    if 31 <= batch <= 60:
        return "Operator"
    if 61 <= batch <= 90:
        return "APC"
    if 91 <= batch <= 100:
        return "Fault"
    return "Unknown"


def make_normal_folds(seed: int = RANDOM_STATE) -> list[list[int]]:
    """Five folds, each containing six batches from every normal regime."""
    rng = np.random.default_rng(seed)
    folds: list[list[int]] = [[] for _ in range(5)]
    for batch_range in (np.arange(1, 31), np.arange(31, 61), np.arange(61, 91)):
        shuffled = rng.permutation(batch_range)
        for fold_index, part in enumerate(np.array_split(shuffled, 5)):
            folds[fold_index].extend(int(value) for value in part)
    return [sorted(fold) for fold in folds]


def _prediction_rows(
    test: pd.DataFrame,
    baseline: Pipeline,
    aware: Pipeline,
    risk_model: Pipeline,
    condition: str,
    fold_name: str,
) -> pd.DataFrame:
    result = test[
        [BATCH_COL, TIME_COL, TARGET_COL, "Fault_Affected", "Fault_Phase"]
    ].copy()
    result["Condition"] = condition
    result["Outer_Fold"] = fold_name
    result["Normal_Only_Prediction"] = baseline.predict(test[MODEL_FEATURES])
    result["Fault_Aware_Prediction"] = aware.predict(test[MODEL_FEATURES])
    result["Fault_Risk_Probability"] = risk_model.predict_proba(
        test[RELIABILITY_FEATURES]
    )[:, 1]
    return result


def evaluate_batch_held_out(
    data: pd.DataFrame,
    max_iter: int = 180,
    fault_multiplier: float = 3.0,
    smoke: bool = False,
) -> pd.DataFrame:
    """Generate honest out-of-fold predictions for normal and fault batches."""
    target_rows = data[TARGET_COL].notna()
    data = data.loc[target_rows].copy()
    pieces: list[pd.DataFrame] = []

    normal_folds = make_normal_folds()
    if smoke:
        normal_folds = normal_folds[:1]

    print("\nNormal-batch outer validation")
    for fold_index, test_batches in enumerate(normal_folds, start=1):
        normal_train_batches = sorted(set(NORMAL_BATCHES) - set(test_batches))
        baseline_train = data[data[BATCH_COL].isin(normal_train_batches)]
        aware_train = data[
            data[BATCH_COL].isin(normal_train_batches + FAULT_BATCHES)
        ]
        test = data[data[BATCH_COL].isin(test_batches)]

        baseline = fit_regressor(
            baseline_train,
            max_iter=max_iter,
            random_state=RANDOM_STATE + fold_index,
            fault_multiplier=1.0,
        )
        aware = fit_regressor(
            aware_train,
            max_iter=max_iter,
            random_state=RANDOM_STATE + 100 + fold_index,
            fault_multiplier=fault_multiplier,
        )
        risk_model = fit_risk_classifier(
            aware_train,
            max_iter=max_iter,
            random_state=RANDOM_STATE + 200 + fold_index,
        )
        pieces.append(
            _prediction_rows(
                test,
                baseline,
                aware,
                risk_model,
                condition="Normal",
                fold_name=f"Normal fold {fold_index}",
            )
        )
        print(
            f"  fold {fold_index}/{len(normal_folds)} complete; "
            f"held out {len(test_batches)} complete normal batches"
        )

    held_out_faults = FAULT_BATCHES[:1] if smoke else FAULT_BATCHES
    print("\nLeave-one-fault-batch-out validation")
    baseline_train = data[data[BATCH_COL].isin(NORMAL_BATCHES)]
    # This baseline is identical for all fault folds and only needs fitting once.
    fault_baseline = fit_regressor(
        baseline_train,
        max_iter=max_iter,
        random_state=RANDOM_STATE + 300,
        fault_multiplier=1.0,
    )

    for fold_index, held_out in enumerate(held_out_faults, start=1):
        development_faults = [batch for batch in FAULT_BATCHES if batch != held_out]
        aware_train = data[
            data[BATCH_COL].isin(NORMAL_BATCHES + development_faults)
        ]
        test = data[data[BATCH_COL].eq(held_out)]
        aware = fit_regressor(
            aware_train,
            max_iter=max_iter,
            random_state=RANDOM_STATE + 400 + fold_index,
            fault_multiplier=fault_multiplier,
        )
        risk_model = fit_risk_classifier(
            aware_train,
            max_iter=max_iter,
            random_state=RANDOM_STATE + 500 + fold_index,
        )
        pieces.append(
            _prediction_rows(
                test,
                fault_baseline,
                aware,
                risk_model,
                condition="Fault",
                fold_name=f"Fault LOBO {held_out}",
            )
        )
        print(
            f"  fault batch {held_out} complete; trained with the other "
            f"{len(development_faults)} fault batches"
        )

    return pd.concat(pieces, ignore_index=True)


In [ ]:
# @title 4E — Calculate metrics, batch bootstrap and empirical error radii
def safe_r2(actual: Sequence[float], predicted: Sequence[float]) -> float:
    actual_array = np.asarray(actual, dtype=float)
    if len(actual_array) < 2 or np.nanstd(actual_array) == 0:
        return float("nan")
    return float(r2_score(actual_array, predicted))


def regression_metrics(
    actual: Sequence[float], predicted: Sequence[float]
) -> dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(math.sqrt(mean_squared_error(actual, predicted))),
        "R2": safe_r2(actual, predicted),
    }


def summarize_predictions(
    predictions: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return pooled row metrics and per-batch metrics for both regressors."""
    model_columns = {
        "Normal-only HGB": "Normal_Only_Prediction",
        "Fault-aware HGB": "Fault_Aware_Prediction",
    }
    overall_records: list[dict[str, Any]] = []
    batch_records: list[dict[str, Any]] = []

    for condition, condition_data in predictions.groupby("Condition", sort=False):
        for model_name, prediction_column in model_columns.items():
            overall_records.append(
                {
                    "Condition": condition,
                    "Model": model_name,
                    "Rows": int(len(condition_data)),
                    "Batches": int(condition_data[BATCH_COL].nunique()),
                    **regression_metrics(
                        condition_data[TARGET_COL],
                        condition_data[prediction_column],
                    ),
                }
            )

            for batch, batch_data in condition_data.groupby(BATCH_COL):
                batch_records.append(
                    {
                        "Condition": condition,
                        "Model": model_name,
                        BATCH_COL: int(batch),
                        "Regime": regime(int(batch)),
                        "Rows": int(len(batch_data)),
                        **regression_metrics(
                            batch_data[TARGET_COL],
                            batch_data[prediction_column],
                        ),
                    }
                )

    return pd.DataFrame(overall_records), pd.DataFrame(batch_records)


def paired_batch_bootstrap(
    batch_metrics: pd.DataFrame,
    condition: str,
    iterations: int = 10000,
    seed: int = RANDOM_STATE,
) -> dict[str, Any]:
    """Bootstrap the paired batch-RMSE difference (aware minus baseline)."""
    subset = batch_metrics[batch_metrics["Condition"].eq(condition)]
    pivot = subset.pivot(index=BATCH_COL, columns="Model", values="RMSE").dropna()
    differences = (
        pivot["Fault-aware HGB"] - pivot["Normal-only HGB"]
    ).to_numpy(float)
    if len(differences) == 0:
        return {"Condition": condition, "Batches": 0}
    rng = np.random.default_rng(seed)
    sampled_means = np.empty(iterations, dtype=float)
    for index in range(iterations):
        sampled_means[index] = rng.choice(
            differences, size=len(differences), replace=True
        ).mean()
    return {
        "Condition": condition,
        "Batches": int(len(differences)),
        "Mean_RMSE_Difference_Aware_Minus_Normal_Only": float(differences.mean()),
        "Median_RMSE_Difference_Aware_Minus_Normal_Only": float(
            np.median(differences)
        ),
        "Bootstrap_95pct_Lower": float(np.quantile(sampled_means, 0.025)),
        "Bootstrap_95pct_Upper": float(np.quantile(sampled_means, 0.975)),
        "Batches_Improved": int((differences < 0).sum()),
        "Proportion_Batches_Improved": float((differences < 0).mean()),
    }


def summarize_fault_risk(predictions: pd.DataFrame) -> pd.DataFrame:
    """Summarize warning performance without treating it as regression accuracy."""
    records: list[dict[str, Any]] = []
    normal = predictions[predictions["Condition"].eq("Normal")]
    fault = predictions[predictions["Condition"].eq("Fault")]

    if len(normal):
        records.append(
            {
                "Measure": "Normal-row warning rate at probability >= 0.5",
                "Value": float(normal["Fault_Risk_Probability"].ge(0.5).mean()),
                "Rows": int(len(normal)),
            }
        )

    if len(fault):
        affected = fault[fault["Fault_Affected"].eq(1)]
        pre_onset = fault[fault["Fault_Affected"].eq(0)]
        records.extend(
            [
                {
                    "Measure": "Fault-affected-row sensitivity at probability >= 0.5",
                    "Value": float(
                        affected["Fault_Risk_Probability"].ge(0.5).mean()
                    ),
                    "Rows": int(len(affected)),
                },
                {
                    "Measure": "Pre-onset warning rate within fault batches",
                    "Value": float(
                        pre_onset["Fault_Risk_Probability"].ge(0.5).mean()
                    ),
                    "Rows": int(len(pre_onset)),
                },
            ]
        )
        if fault["Fault_Affected"].nunique() == 2:
            records.append(
                {
                    "Measure": "Fault-phase ROC AUC within held-out fault batches",
                    "Value": float(
                        roc_auc_score(
                            fault["Fault_Affected"],
                            fault["Fault_Risk_Probability"],
                        )
                    ),
                    "Rows": int(len(fault)),
                }
            )

    return pd.DataFrame(records)


def higher_quantile(values: Iterable[float], quantile: float) -> float:
    array = np.asarray(list(values), dtype=float)
    try:
        return float(np.quantile(array, quantile, method="higher"))
    except TypeError:
        return float(np.quantile(array, quantile, interpolation="higher"))


def interval_radii(predictions: pd.DataFrame, coverage: float = 0.90) -> dict[str, float]:
    """Estimate descriptive absolute-error radii from held-out predictions."""
    radii: dict[str, float] = {}
    for condition in ("Normal", "Fault"):
        subset = predictions[predictions["Condition"].eq(condition)]
        if len(subset):
            errors = (
                subset[TARGET_COL] - subset["Fault_Aware_Prediction"]
            ).abs()
            radii[condition.lower()] = higher_quantile(errors, coverage)
    if "normal" not in radii:
        radii["normal"] = float("nan")
    if "fault" not in radii:
        radii["fault"] = radii["normal"]
    radii["coverage_target"] = float(coverage)
    return radii


In [ ]:
# @title 4F — OOD reference and the final sensor bundle
def _balanced_normal_sample(data: pd.DataFrame, rows_per_batch: int = 250) -> pd.DataFrame:
    pieces = []
    normal = data[data[BATCH_COL].isin(NORMAL_BATCHES)]
    for _, batch_data in normal.groupby(BATCH_COL, sort=True):
        count = min(rows_per_batch, len(batch_data))
        positions = np.linspace(0, len(batch_data) - 1, count, dtype=int)
        pieces.append(batch_data.iloc[positions])
    return pd.concat(pieces, ignore_index=True)


def fit_ood_reference(data: pd.DataFrame) -> dict[str, Any]:
    """Fit a normal-data similarity model and a conservative score threshold."""
    sample = _balanced_normal_sample(data)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    pca = PCA(n_components=0.95, svd_solver="full")
    isolation_forest = IsolationForest(
        n_estimators=400,
        contamination="auto",
        max_samples="auto",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    transformed = imputer.fit_transform(sample[RELIABILITY_FEATURES])
    transformed = scaler.fit_transform(transformed)
    transformed = pca.fit_transform(transformed)
    isolation_forest.fit(transformed)
    scores = -isolation_forest.score_samples(transformed)
    threshold = higher_quantile(scores, 0.99)

    return {
        "features": list(RELIABILITY_FEATURES),
        "imputer": imputer,
        "scaler": scaler,
        "pca": pca,
        "isolation_forest": isolation_forest,
        "threshold": threshold,
        "threshold_quantile": 0.99,
        "training_rows": int(len(sample)),
    }


def score_ood(bundle: dict[str, Any], frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    ood = bundle["ood"]
    transformed = ood["imputer"].transform(frame[ood["features"]])
    transformed = ood["scaler"].transform(transformed)
    transformed = ood["pca"].transform(transformed)
    scores = -ood["isolation_forest"].score_samples(transformed)
    flags = scores > float(ood["threshold"])
    return scores, flags


def fit_deployable_bundle(
    data: pd.DataFrame,
    radii: dict[str, float],
    max_iter: int,
    fault_multiplier: float,
) -> dict[str, Any]:
    """Fit the model that can be saved after evaluation is complete."""
    train = data[data[TARGET_COL].notna()].copy()
    regressor = fit_regressor(
        train,
        max_iter=max_iter,
        random_state=RANDOM_STATE + 900,
        fault_multiplier=fault_multiplier,
    )
    risk_model = fit_risk_classifier(
        train,
        max_iter=max_iter,
        random_state=RANDOM_STATE + 901,
    )
    return {
        "format_version": "1.0",
        "purpose": "Current penicillin concentration soft sensor with reliability warnings",
        "regressor": regressor,
        "risk_classifier": risk_model,
        "ood": fit_ood_reference(train),
        "model_features": list(MODEL_FEATURES),
        "reliability_features": list(RELIABILITY_FEATURES),
        "interval_radii": radii,
        "fault_multiplier": float(fault_multiplier),
        "normal_batches": list(NORMAL_BATCHES),
        "fault_batches": list(FAULT_BATCHES),
        "target_column": TARGET_COL,
        "time_column": TIME_COL,
        "batch_column": BATCH_COL,
        "software": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
        },
    }


def predict_with_bundle(bundle: dict[str, Any], feature_frame: pd.DataFrame) -> pd.DataFrame:
    """Return concentration, descriptive interval, and reliability warnings."""
    prediction = bundle["regressor"].predict(feature_frame[MODEL_FEATURES])
    risk = bundle["risk_classifier"].predict_proba(
        feature_frame[RELIABILITY_FEATURES]
    )[:, 1]
    ood_score, ood_flag = score_ood(bundle, feature_frame)

    normal_radius = float(bundle["interval_radii"]["normal"])
    fault_radius = float(bundle["interval_radii"]["fault"])
    radius = normal_radius + risk * (fault_radius - normal_radius)
    radius = np.where(ood_flag, radius * 1.25, radius)

    trust = np.select(
        [ood_flag & (risk >= 0.5), ood_flag | (risk >= 0.5)],
        ["Low: OOD and high fault risk", "Caution: one warning active"],
        default="Higher: no warning active",
    )

    result = feature_frame[[BATCH_COL, TIME_COL]].copy()
    result["Penicillin_Prediction_g_L"] = prediction
    result["Empirical_Lower_g_L"] = np.maximum(0.0, prediction - radius)
    result["Empirical_Upper_g_L"] = prediction + radius
    result["Fault_Risk_Probability"] = risk
    result["OOD_Score"] = ood_score
    result["OOD_Threshold"] = float(bundle["ood"]["threshold"])
    result["OOD_Flag"] = ood_flag
    result["Trust_Status"] = trust
    return result


def deployment_reliability_by_batch(
    bundle: dict[str, Any], feature_frame: pd.DataFrame
) -> pd.DataFrame:
    """Describe warnings from the final bundle without reporting in-sample error."""
    risk = bundle["risk_classifier"].predict_proba(
        feature_frame[RELIABILITY_FEATURES]
    )[:, 1]
    ood_score, ood_flag = score_ood(bundle, feature_frame)
    diagnostic = feature_frame[[BATCH_COL]].copy()
    diagnostic["Fault_Risk_Probability"] = risk
    diagnostic["OOD_Score"] = ood_score
    diagnostic["OOD_Flag"] = ood_flag

    summary = (
        diagnostic.groupby(BATCH_COL, as_index=False)
        .agg(
            Rows=(BATCH_COL, "size"),
            Median_Fault_Risk=("Fault_Risk_Probability", "median"),
            Maximum_Fault_Risk=("Fault_Risk_Probability", "max"),
            Fault_Warning_Rate=(
                "Fault_Risk_Probability",
                lambda values: float(values.ge(0.5).mean()),
            ),
            Median_OOD_Score=("OOD_Score", "median"),
            OOD_95pct_Score=("OOD_Score", lambda values: float(values.quantile(0.95))),
            OOD_Flag_Rate=("OOD_Flag", "mean"),
        )
    )
    summary["Documented_Group"] = np.where(
        summary[BATCH_COL].isin(FAULT_BATCHES), "Fault", "Normal"
    )
    summary["OOD_Threshold"] = float(bundle["ood"]["threshold"])
    return summary


In [ ]:
# @title 4G — Figures, result saving and run functions
def create_figures(
    overall: pd.DataFrame,
    batch_metrics: pd.DataFrame,
    predictions: pd.DataFrame,
    output: Path,
) -> None:
    sns.set_theme(style="whitegrid")

    plt.figure(figsize=(8, 5))
    sns.barplot(data=overall, x="Condition", y="RMSE", hue="Model")
    plt.ylabel("Pooled RMSE (g/L)")
    plt.title("Normal-only versus fault-aware soft sensor")
    plt.tight_layout()
    plt.savefig(output / "figure_1_rmse_comparison.png", dpi=300)
    plt.close()

    fault_batch = batch_metrics[batch_metrics["Condition"].eq("Fault")]
    if len(fault_batch):
        plt.figure(figsize=(11, 5))
        sns.barplot(data=fault_batch, x=BATCH_COL, y="RMSE", hue="Model")
        plt.ylabel("Batch RMSE (g/L)")
        plt.title("Leave-one-fault-batch-out error")
        plt.tight_layout()
        plt.savefig(output / "figure_2_fault_batch_rmse.png", dpi=300)
        plt.close()

    fault_predictions = predictions[predictions["Condition"].eq("Fault")]
    if len(fault_predictions):
        figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
        for axis, (title, column) in zip(
            axes,
            [
                ("Normal-only", "Normal_Only_Prediction"),
                ("Fault-aware", "Fault_Aware_Prediction"),
            ],
        ):
            axis.scatter(
                fault_predictions[TARGET_COL],
                fault_predictions[column],
                s=8,
                alpha=0.25,
            )
            minimum = float(
                min(fault_predictions[TARGET_COL].min(), fault_predictions[column].min())
            )
            maximum = float(
                max(fault_predictions[TARGET_COL].max(), fault_predictions[column].max())
            )
            axis.plot([minimum, maximum], [minimum, maximum], "r--", linewidth=1)
            axis.set_title(title)
            axis.set_xlabel("Actual penicillin (g/L)")
            axis.set_ylabel("Predicted penicillin (g/L)")
        figure.suptitle("Held-out fault batches: actual versus predicted")
        figure.tight_layout()
        figure.savefig(output / "figure_3_fault_actual_vs_predicted.png", dpi=300)
        plt.close(figure)

        plt.figure(figsize=(9, 5))
        sns.boxplot(
            data=predictions,
            x="Fault_Phase",
            y="Fault_Risk_Probability",
            order=["Normal", "Fault batch: before onset", "Fault batch: onset/after"],
        )
        plt.xticks(rotation=10)
        plt.xlabel("")
        plt.ylabel("Out-of-fold fault-risk probability")
        plt.title("Reliability warning by process phase")
        plt.tight_layout()
        plt.savefig(output / "figure_4_fault_risk_by_phase.png", dpi=300)
        plt.close()


def _json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.ndarray):
        return [_json_safe(item) for item in value.tolist()]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_readable_summary(
    path: Path,
    mode: str,
    overall: pd.DataFrame,
    bootstrap: pd.DataFrame,
    risk: pd.DataFrame,
    radii: dict[str, float],
) -> None:
    lines = [
        "INDPENSIM BATCH-HELD-OUT FAULT-AWARE SOFT SENSOR",
        "=" * 54,
        "",
        f"Run mode: {mode}",
        "Target: current penicillin concentration (not future forecasting)",
        "",
        "Regression results",
        overall.to_string(index=False),
        "",
        "Paired complete-batch comparison",
        bootstrap.to_string(index=False),
        "",
        "Fault-risk warning results",
        risk.to_string(index=False),
        "",
        "Descriptive interval radii",
        json.dumps(_json_safe(radii), indent=2),
        "",
        "How to interpret this run",
        "- A negative aware-minus-normal-only RMSE difference favours the fault-aware model.",
        "- The bootstrap interval is across complete batches, not individual time rows.",
        "- Fault-risk and OOD are reliability warnings; neither proves a physical fault.",
        "- The empirical interval is descriptive and is not a formal safety guarantee.",
        "- Only a full run covering all 5 normal folds and all 10 fault folds is reportable.",
    ]
    if mode == "smoke":
        lines.extend(
            [
                "",
                "IMPORTANT: SMOKE MODE IS ONLY A TECHNICAL INSTALLATION CHECK.",
                "Do not cite its metrics in a paper or report.",
            ]
        )
    path.write_text("\n".join(lines), encoding="utf-8")


def run_training(
    data_path: str | Path,
    output_dir: str | Path = "fault_aware_results",
    mode: str = "full",
    max_iter: int | None = None,
    fault_multiplier: float = 3.0,
) -> dict[str, Path]:
    """Run evaluation, fit the deployable bundle, and save all outputs."""
    if mode not in {"full", "smoke"}:
        raise ValueError("run_training mode must be 'full' or 'smoke'.")
    if fault_multiplier <= 0:
        raise ValueError("fault_multiplier must be positive.")

    output = Path(output_dir).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    if max_iter is None:
        max_iter = 50 if mode == "smoke" else 180

    start = time.time()
    print("Loading process variables (Raman spectra are intentionally skipped)...")
    data = load_indpensim(data_path, require_target=True)
    validate_training_data(data)
    print(f"Loaded {len(data):,} rows from {data[BATCH_COL].nunique()} batches.")
    print("Engineering within-batch causal history features...")
    data = engineer_features(data)
    data = add_fault_phase_labels(data)

    predictions = evaluate_batch_held_out(
        data,
        max_iter=max_iter,
        fault_multiplier=fault_multiplier,
        smoke=(mode == "smoke"),
    )
    overall, batch_metrics = summarize_predictions(predictions)
    bootstrap_conditions = ["Normal", "Fault"]
    bootstrap = pd.DataFrame(
        [
            paired_batch_bootstrap(batch_metrics, condition=condition)
            for condition in bootstrap_conditions
            if condition in set(batch_metrics["Condition"])
        ]
    )
    risk = summarize_fault_risk(predictions)
    radii = interval_radii(predictions, coverage=0.90)

    predictions.to_csv(output / "cross_validated_predictions.csv", index=False)
    overall.to_csv(output / "cross_validated_overall_metrics.csv", index=False)
    batch_metrics.to_csv(output / "cross_validated_batch_metrics.csv", index=False)
    bootstrap.to_csv(output / "paired_batch_bootstrap.csv", index=False)
    risk.to_csv(output / "fault_risk_summary.csv", index=False)

    create_figures(overall, batch_metrics, predictions, output)

    print("\nFitting the deployable model on all 100 labelled development batches...")
    bundle = fit_deployable_bundle(
        data,
        radii=radii,
        max_iter=max_iter,
        fault_multiplier=fault_multiplier,
    )
    model_path = output / "fault_aware_soft_sensor.joblib"
    joblib.dump(bundle, model_path, compress=3)
    deployment_reliability_by_batch(bundle, data).to_csv(
        output / "deployment_reliability_by_batch.csv", index=False
    )

    metadata = {
        "run_mode": mode,
        "reportable": mode == "full",
        "rows": int(len(data)),
        "batches": int(data[BATCH_COL].nunique()),
        "normal_outer_folds_completed": 1 if mode == "smoke" else 5,
        "fault_outer_folds_completed": 1 if mode == "smoke" else 10,
        "regressor": "HistGradientBoostingRegressor",
        "fault_risk_model": "HistGradientBoostingClassifier",
        "fault_weight_multiplier": float(fault_multiplier),
        "max_iter": int(max_iter),
        "model_features": list(MODEL_FEATURES),
        "reliability_features": list(RELIABILITY_FEATURES),
        "interval_radii": radii,
        "ood_training_rows": bundle["ood"]["training_rows"],
        "ood_threshold": bundle["ood"]["threshold"],
        "elapsed_minutes": float((time.time() - start) / 60.0),
        "software": bundle["software"],
        "scientific_scope": [
            "Current-time penicillin concentration estimation",
            "IndPenSim in-silico batches 1--100",
            "Batch-held-out evaluation",
            "Fault-risk and OOD warnings are not causal fault diagnoses",
        ],
    }
    (output / "run_metadata.json").write_text(
        json.dumps(_json_safe(metadata), indent=2), encoding="utf-8"
    )
    write_readable_summary(
        output / "RESULTS_SUMMARY.txt",
        mode=mode,
        overall=overall,
        bootstrap=bootstrap,
        risk=risk,
        radii=radii,
    )

    if mode == "smoke":
        (output / "SMOKE_TEST_ONLY.txt").write_text(
            "This incomplete run verifies the software path only. "
            "Run --mode full before reporting results.\n",
            encoding="utf-8",
        )

    print("\nOverall held-out metrics")
    print(overall.to_string(index=False))
    print(f"\nSaved outputs to: {output}")
    return {
        "output_dir": output,
        "model": model_path,
        "summary": output / "RESULTS_SUMMARY.txt",
    }


def run_prediction(
    data_path: str | Path,
    model_path: str | Path,
    output_dir: str | Path,
) -> Path:
    """Apply a previously saved bundle to one or more new batch trajectories."""
    output = Path(output_dir).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    bundle = joblib.load(Path(model_path).expanduser().resolve())
    data = load_indpensim(data_path, require_target=False)
    data = engineer_features(data)
    result = predict_with_bundle(bundle, data)
    path = output / "new_batch_soft_sensor_predictions.csv"
    result.to_csv(path, index=False)
    print(f"Saved {len(result):,} predictions to: {path}")
    return path


## Step 5 — Run the experiment

This cell performs fitting and held-out evaluation, then saves the final bundle. Runtime depends on your Colab CPU and Drive read speed. Keep the session connected until it finishes. The cell prints progress as each outer fold completes.

Changing algorithms after inspecting these outer results makes subsequent comparisons exploratory; freeze the protocol or use nested validation before making confirmatory claims.


In [ ]:
# @title Step 5 — Train, evaluate and save the fault-aware sensor
artifacts = run_training(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    mode=RUN_MODE,
    max_iter=None,
    fault_multiplier=3.0,
)
print("Completed. Results:", artifacts["output_dir"])
print("Saved sensor:", artifacts["model"])


## Step 6 — Read the result tables

Lower MAE and RMSE are better. R² is not an accuracy percentage; a negative batch R² means the predictions are worse than using that batch's mean as a retrospective constant reference.

The percentage reduction below is positive when the fault-aware model improves and negative when it worsens. Check both pooled scores and individual batches. New fold schemes and training sizes mean these values need not match the old fixed 15-normal/10-fault test tables.


In [ ]:
# @title Step 6 — Overall, per-batch and warning results
from IPython.display import display, Markdown, Image

results_dir = Path(artifacts["output_dir"])
overall_results = pd.read_csv(results_dir / "cross_validated_overall_metrics.csv")
batch_results = pd.read_csv(results_dir / "cross_validated_batch_metrics.csv")
paired_results = pd.read_csv(results_dir / "paired_batch_bootstrap.csv")
risk_results = pd.read_csv(results_dir / "fault_risk_summary.csv")

if RUN_MODE == "smoke":
    display(Markdown("**SMOKE CHECK ONLY — these incomplete metrics are not report results.**"))

display(Markdown("### Pooled held-out results"))
display(overall_results.round(4))

reductions = []
for condition, group in overall_results.groupby("Condition", sort=False):
    lookup = group.set_index("Model")
    baseline_row = lookup.loc["Normal-only HGB"]
    aware_row = lookup.loc["Fault-aware HGB"]
    reductions.append({
        "Condition": condition,
        "MAE_Reduction_Percent": 100.0 * (1 - aware_row["MAE"] / baseline_row["MAE"]),
        "RMSE_Reduction_Percent": 100.0 * (1 - aware_row["RMSE"] / baseline_row["RMSE"]),
    })
display(pd.DataFrame(reductions).round(2))

display(Markdown("### Each held-out fault batch"))
fault_batch_table = batch_results.loc[batch_results["Condition"].eq("Fault")]
display(fault_batch_table.pivot(index=BATCH_COL, columns="Model", values=["MAE", "RMSE", "R2"]).round(4))

display(Markdown("### Paired complete-batch RMSE differences"))
display(paired_results.round(4))
print("Negative aware-minus-normal-only differences favour the fault-aware model.")
print("The bootstrap is descriptive across batches; overlapping fold training sets are not independent campaigns.")

display(Markdown("### Fault-risk warning performance"))
display(risk_results.round(4))
print("Warning rates and sensitivity are fractions; multiply by 100 to obtain percentages.")
print("A fault-risk score or OOD flag is not proof of a physical fault.")


## Step 7 — View the four output figures

The first two figures show error comparisons. The third shows how closely predictions follow the measured target. The fourth describes the separate fault-risk warning, not concentration accuracy.


In [ ]:
# @title Step 7 — Display the saved graphs
figure_captions = [
    ("figure_1_rmse_comparison.png", "Grouped bar chart: pooled normal and fault RMSE. A shorter bar means a smaller error."),
    ("figure_2_fault_batch_rmse.png", "Grouped bar chart: RMSE for each held-out fault batch. Use this to find batches that improved and those that still fail."),
    ("figure_3_fault_actual_vs_predicted.png", "Scatter plots: actual versus predicted concentration on held-out fault batches. Points closer to the diagonal are better."),
    ("figure_4_fault_risk_by_phase.png", "Box plots: fault-risk scores for normal, pre-onset and onset/after-onset rows. This describes warning behaviour, not regression accuracy."),
]
for filename, caption in figure_captions:
    figure_path = results_dir / filename
    if figure_path.exists():
        display(Image(filename=str(figure_path), width=1050))
        display(Markdown(caption))


## Step 8 — Inspect one batch over time

This line graph uses the saved **out-of-fold** predictions, not predictions from the final all-batch fitted model. Start with batch 100, which remained difficult in the previous evaluation. In smoke mode only batch 91 and one normal fold are available.


In [ ]:
# @title Step 8 — Plot a held-out batch trajectory
BATCH_TO_PLOT = 100 # @param {type:"integer"}

oof_predictions = pd.read_csv(results_dir / "cross_validated_predictions.csv")
trajectory = oof_predictions.loc[oof_predictions[BATCH_COL].eq(BATCH_TO_PLOT)].sort_values(TIME_COL)

if trajectory.empty:
    print("This batch was not evaluated in the current run. Available batches:")
    print(sorted(oof_predictions[BATCH_COL].unique().tolist()))
else:
    figure, axis = plt.subplots(figsize=(11, 5))
    axis.plot(trajectory[TIME_COL], trajectory[TARGET_COL], color="black", label="Actual", linewidth=2)
    axis.plot(trajectory[TIME_COL], trajectory["Normal_Only_Prediction"], label="Normal-only HGB", alpha=0.8)
    axis.plot(trajectory[TIME_COL], trajectory["Fault_Aware_Prediction"], label="Fault-aware HGB", alpha=0.9)
    affected = trajectory.loc[trajectory["Fault_Affected"].eq(1), TIME_COL]
    if len(affected):
        axis.axvline(float(affected.min()), color="firebrick", linestyle="--", label="Recorded fault onset")
    axis.set(xlabel="Time (h)", ylabel="Penicillin concentration (g/L)", title=f"Batch {BATCH_TO_PLOT}: held-out predictions")
    axis.legend()
    figure.tight_layout()
    trajectory_path = results_dir / f"batch_{BATCH_TO_PLOT}_held_out_trajectory.png"
    figure.savefig(trajectory_path, dpi=220, bbox_inches="tight")
    plt.close(figure)
    display(Image(filename=str(trajectory_path), width=1050))
    print("This graph is a time-series line plot. The onset line is for evaluation only, not a prediction input.")


## Step 9 — Save a results ZIP

This cell creates an archive next to the run directory. Set `DOWNLOAD_RESULTS_ZIP = True` to request a Colab download; outside Colab, use the printed local path.


In [ ]:
# Create a ZIP in Colab or local Jupyter. Downloading is optional.
import shutil

DOWNLOAD_RESULTS_ZIP = False
archive_path = shutil.make_archive(
    base_name=str(results_dir) + "_results",
    format="zip",
    root_dir=str(results_dir),
)
print("Results remain saved in:", results_dir)
print("ZIP saved at:", archive_path)
if DOWNLOAD_RESULTS_ZIP and IN_GOOGLE_COLAB:
    from google.colab import files
    files.download(archive_path)
elif DOWNLOAD_RESULTS_ZIP:
    print("Outside Colab, open the saved ZIP path in your file manager.")


## Optional — Apply the final sensor to a new batch

Leave `RUN_NEW_BATCH_PREDICTION` off unless you have a new batch CSV. Supply the same 20 input column names and units, with all observations from the start of the batch through the current time; the history and cumulative-feed features need that observed prefix. A single isolated current row is not equivalent.

This step does not retrain the model. No penicillin target or fault-reference column is required for prediction. `Batch_ID`, if absent, is reconstructed from time resets. A whole batch must not share one ID with a different batch.

Use a bundle from a **full** run. The new CSV is not considered an independent accuracy test merely because you rename an old training CSV. Fault-risk scores and the interval remain provisional; a missing warning does not guarantee a correct estimate.


In [ ]:
# # @title Optional — Predict a new batch (off by default)
# RUN_NEW_BATCH_PREDICTION = False # @param {type:"boolean"}
# NEW_BATCH_CSV = "" # @param {type:"string"}

# if not RUN_NEW_BATCH_PREDICTION:
#     print("Optional new-batch prediction skipped.")
# else:
#     if RUN_MODE != "full":
#         raise ValueError("Run the full evaluation first; do not use a smoke-mode sensor here.")
#     if not NEW_BATCH_CSV.strip() or not Path(NEW_BATCH_CSV).is_file():
#         raise FileNotFoundError("Set NEW_BATCH_CSV to the complete path of your new batch CSV.")
#     new_prediction_dir = results_dir / "new_batch_predictions"
#     prediction_csv = run_prediction(
#         data_path=NEW_BATCH_CSV,
#         model_path=artifacts["model"],
#         output_dir=new_prediction_dir,
#     )
#     display(pd.read_csv(prediction_csv).head(20))
#     print("Prediction output:", prediction_csv)
#     print("If you want these new predictions included in the ZIP, rerun Step 9.")


## Reporting checklist

- Use `cross_validated_overall_metrics.csv` and `cross_validated_batch_metrics.csv` for accuracy claims, not final-model fitted predictions.
- State that fault evaluation is leave-one-fault-batch-out, with nine other fault batches available for training. It does not prove performance on entirely new fault mechanisms.
- Report any negative batch R², particularly the remaining difficult batches, even if pooled R² is high.
- Report fault-risk warnings separately from concentration errors. The class-weighted risk score is not a calibrated probability of a real fault.
- The OOD threshold is fitted on normal reference data and is a similarity check, not an independently validated fault decision rule.
- The empirical range interpolates held-out normal/fault error quantiles using the risk score and widens by 25% when OOD is flagged. This heuristic has no guaranteed 90% coverage, especially under new faults.
- Keep the run configuration and software versions from `run_metadata.json`. Full mode completes the planned computation; it does not by itself certify publication readiness.
- All results are from simulated IndPenSim fermentation, and this model estimates current concentration rather than forecasting future concentration.

Dataset: Goldrick, S. (2019), *Data for: Modern day monitoring and control challenges outlined on an industrial-scale benchmark fermentation process*, version 1. [DOI: 10.17632/pdnjz7zz5x.1](https://doi.org/10.17632/pdnjz7zz5x.1). Follow the dataset's CC BY 4.0 attribution terms.

This repository copy changes setup paths and clears saved outputs, not the underlying modelling calculations. Archived numerical results are provided separately under `results/ml4/`.
